# Test du comportement de `IndexRegularizer`

Ce notebook vérifie les méthodes `is_regular` et `regularize` sur quatre jeux de données :

| # | Type | Régularité |
|---|------|------------|
| 1 | Série temporelle | Régulière |
| 2 | Série temporelle | Irrégulière (mensuel + trimestriel) |
| 3 | Panel | Régulier |
| 4 | Panel | Irrégulier (mensuel + trimestriel) |

In [ ]:
# Importation des modules
# Modules de base
import pandas as pd
import numpy as np
import sys

# Ajout du chemin
sys.path.append('..')

# Importation de la classe du package
from tsforecast.frequency.regularizer import IndexRegularizer

# Instanciation de la classe à tester
regularizer = IndexRegularizer()

## 1. Série temporelle régulière

In [ ]:
# Génération d'une série mensuelle régulière
dates_regular = pd.date_range("2020-01-01", periods=24, freq="MS")
ts_regular = pd.DataFrame(
    {"value": np.random.default_rng(42).normal(size=24)},
    index=dates_regular,
)
ts_regular.head()

In [ ]:
# Vérification de la régularité → True attendu
print("is_regular :", regularizer.is_regular(ts_regular))

In [ ]:
# Régularisation (aucun changement attendu)
ts_regular_out = regularizer.regularize(ts_regular)
print(f"Longueur avant : {len(ts_regular)}, après : {len(ts_regular_out)}")
ts_regular_out.head()

## 2. Série temporelle irrégulière

Construction par concaténation d'une colonne trimestrielle (démarrant plus tôt)
et d'une colonne mensuelle, puis empilement en une seule `Series` pour créer
un index irrégulier.

In [ ]:
rng = np.random.default_rng(0)

# Colonne trimestrielle : Q1-2019 → Q4-2019 (débute avant la mensuelle)
dates_quarterly = pd.date_range("2019-01-01", periods=8, freq="QS")
quarterly = pd.Series(rng.normal(size=8), index=dates_quarterly, name="quarterly_value")

# Colonne mensuelle : jan-2020 → déc-2020
dates_monthly = pd.date_range("2020-01-01", periods=12, freq="MS")
monthly = pd.Series(rng.normal(size=12), index=dates_monthly, name="monthly_value")

# Concaténation → index irrégulier (pas trimestriels puis pas mensuels)
ts_irregular = pd.concat([quarterly.to_frame(), monthly.to_frame()], axis=1)
print("Index :")
print(ts_irregular.index)

In [ ]:
ts_irregular.head(10)

In [ ]:
# Vérification de la régularité → False attendu
print("is_regular :", regularizer.is_regular(ts_irregular))

In [ ]:
# Régularisation → remplissage des trous à la fréquence détectée
ts_irregular_out = regularizer.regularize(ts_irregular)
print(f"Longueur avant : {len(ts_irregular)}, après : {len(ts_irregular_out)}")
print("\nSérie régularisée :")
ts_irregular_out

In [ ]:
# Vérification post-régularisation → True attendu
print("is_regular après régularisation :", regularizer.is_regular(ts_irregular_out))

## 3. Panel régulier

In [ ]:
# Génération d'un panel mensuel régulier à deux entités
rng = np.random.default_rng(1)
dates_panel = pd.date_range("2021-01-01", periods=12, freq="MS")
entities = ["FR", "DE"]

frames = []
for entity in entities:
    df_tmp = pd.DataFrame(
        {"value": rng.normal(size=12)},
        index=pd.MultiIndex.from_arrays(
            [np.full(12, entity), dates_panel],
            names=["country", "date"],
        ),
    )
    frames.append(df_tmp)

panel_regular = pd.concat(frames)
panel_regular

In [ ]:
# Vérification de la régularité globale → True attendu
print("is_regular (global) :", regularizer.is_regular(panel_regular))
print("is_regular (per_entity) :", regularizer.is_regular(panel_regular, per_entity=True))

In [ ]:
# Régularisation (aucun changement attendu)
panel_regular_out = regularizer.regularize(panel_regular)
print(f"Longueur avant : {len(panel_regular)}, après : {len(panel_regular_out)}")
panel_regular_out

## 4. Panel irrégulier

Même principe que pour la série temporelle irrégulière, appliqué à chaque entité :
une partie trimestrielle (démarrant plus tôt) concaténée avec une partie mensuelle.

In [ ]:
rng = np.random.default_rng(2)

frames_irreg = []
for entity in entities:
    # Partie trimestrielle : Q1-2019 → Q4-2019
    dq = pd.date_range("2019-01-01", periods=8, freq="QS")
    # Partie mensuelle : jan-2020 → déc-2020
    dm = pd.date_range("2020-01-01", periods=12, freq="MS")
    # Longueur des dates
    nq = len(dq)
    nm = len(dm)
    # Création du jeu de données mensuel
    df_monthly_tmp = pd.DataFrame(
        {"monthly_value": rng.normal(size=nm)},
        index=pd.MultiIndex.from_arrays(
            [np.full(nm, entity), dm],
            names=["country", "date"],
        ),
    )
    # Création du jeu de données trimestriel
    df_quarterly_tmp = pd.DataFrame(
        {"quarterly_value": rng.normal(size=nq)},
        index=pd.MultiIndex.from_arrays(
            [np.full(nq, entity), dq],
            names=["country", "date"],
        ),
    )
    # Concaténation
    df_tmp = pd.concat([df_monthly_tmp, df_quarterly_tmp], axis=1)
    frames_irreg.append(df_tmp)

panel_irregular = pd.concat(frames_irreg)
panel_irregular

In [ ]:
# Vérification de la régularité → False attendu
print("is_regular (global) :", regularizer.is_regular(panel_irregular))
print("is_regular (per_entity) :", regularizer.is_regular(panel_irregular, per_entity=True))

In [ ]:
# Régularisation → remplissage des trous pour chaque entité
panel_irregular_out = regularizer.regularize(panel_irregular)
print(f"Longueur avant : {len(panel_irregular)}, après : {len(panel_irregular_out)}")
panel_irregular_out

In [ ]:
# Vérification post-régularisation → True attendu
print(
    "is_regular après régularisation (global) :",
    regularizer.is_regular(panel_irregular_out),
)
print(
    "is_regular après régularisation (per_entity) :",
    regularizer.is_regular(panel_irregular_out, per_entity=True),
)